[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C53_RealTime_Detectors_Course/02_label_assignment/02_label_assignment.ipynb)

# 02 · 标签分配（ATSS / SimOTA / TaskAligned / dynamic-k / 去冲突）

目标：在**同一组合成 TSR 数据**上，从零实现四种标签分配策略，
并用数字回答一个问题——**它们到底把哪些格点标成了正样本，差别有多大**。

本 notebook 你会亲手实现：

1. 一个 640×640 / stride 8·16·32 的 FPN 格点系统与一组 TSR 风格的 GT（16px→96px）
2. 一个「训练到一半」的**合成检测器**（分类分数与定位质量**部分解耦**——这正是 TaskAligned 要治的病）
3. **MaxIoU 静态分配** + 「可匹配尺寸区间」的解析推导
4. **ATSS**：每层 top-k 候选 + 自适应阈值 `mean + std`
5. **SimOTA 完整版**：几何先验 → 代价矩阵 → **dynamic-k** → **去冲突**
6. **TaskAligned**：$t = s^{\alpha}u^{\beta}$ 的选择与仲裁
7. 四种分配的**正样本集合差异**（数量 / 层分布 / 质量 / Jaccard 重叠）

> 心智模型：**网络结构决定模型能表达什么，标签分配决定模型实际学到什么。**

## 1 · 合成一个 TSR 风格的检测场景

640×640 输入、三层 FPN。GT 按真实交通标志的成像尺寸设计：
远处 16px 的限速牌、紧贴其下的 18px 辅助牌（**密集构型**）、
中距离 24px 警告牌、48px 禁令牌、近处 96px 指示牌。

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

IMG = 640
STRIDES = [8, 16, 32]        # P3 / P4 / P5
ANCHOR_SCALE = 4.0           # 每个格点一个 anchor，边长 = ANCHOR_SCALE * stride

def make_points(img=IMG, strides=STRIDES):
    '''FPN 各层的格点中心（图像像素坐标）+ 每个格点的 stride 与层号。'''
    P, S, L = [], [], []
    for li, s in enumerate(strides):
        n = img // s
        gy, gx = np.meshgrid(np.arange(n), np.arange(n), indexing='ij')
        P.append(np.stack([(gx.ravel() + 0.5) * s, (gy.ravel() + 0.5) * s], 1).astype(float))
        S.append(np.full(n * n, float(s)))
        L.append(np.full(n * n, li))
    return np.concatenate(P), np.concatenate(S), np.concatenate(L)

POINTS, PT_S, PT_L = make_points()
ANCHORS = np.concatenate([POINTS - (ANCHOR_SCALE * PT_S[:, None]) / 2,
                          POINTS + (ANCHOR_SCALE * PT_S[:, None]) / 2], 1)   # xyxy

# 一个典型 TSR 场景（尺寸对应 ~80m / ~75m / ~55m / ~28m / ~14m）
GT = np.array([
    [300., 180., 316., 196.],   # 0  16x16  远处限速牌
    [305., 200., 323., 218.],   # 1  18x18  紧贴其下的辅助牌 <- 密集构型
    [352., 172., 376., 196.],   # 2  24x24  警告牌
    [180., 300., 228., 348.],   # 3  48x48  禁令牌
    [ 60., 260., 156., 356.],   # 4  96x96  近处指示牌
])
GT_CLS = np.array([0, 2, 1, 0, 2])      # 0=限速 1=警告 2=指示
N_CLS = 3
GT_WH = GT[:, 2:] - GT[:, :2]

print(f'FPN 格点总数 {len(POINTS)}  = 80^2 + 40^2 + 20^2 = {80**2 + 40**2 + 20**2}')
print(f'各层格点数: ' + ', '.join(f'stride{s}:{int((PT_S == s).sum())}' for s in STRIDES))
print()
print(f'{"GT":>3s} {"类":>3s} {"尺寸":>10s} {"面积":>7s} {"中心":>16s}')
for i, b in enumerate(GT):
    print(f'{i:>3d} {GT_CLS[i]:>3d} {GT_WH[i,0]:>4.0f}x{GT_WH[i,1]:<5.0f} '
          f'{GT_WH[i].prod():>7.0f} ({(b[0]+b[2])/2:>7.1f},{(b[1]+b[3])/2:>6.1f})')

assert len(POINTS) == 8400 and ANCHORS.shape == (8400, 4)
assert (ANCHORS[:, 2] - ANCHORS[:, 0]).min() == 32.0    # stride8 -> 32px anchor
assert (ANCHORS[:, 2] - ANCHORS[:, 0]).max() == 128.0   # stride32 -> 128px anchor
print()
print('✅ 场景就位：5 个 GT，面积跨度 36 倍（16px vs 96px），含一对紧贴的密集标志。')

## 2 · IoU、中心先验，与一个「训练到一半」的合成检测器

SimOTA 与 TaskAligned 都是**预测感知**的——它们要看模型当前预测得怎么样。
所以我们需要一个合成检测器。合成规则（物理上合理）：

- 格点离某个 GT 中心越近（按 GT 尺寸归一化），该位置的**回归越准、分类分数越高**；
- 额外注入一份**与定位无关**的分类噪声 —— 于是「分数高」与「框准」只是**部分相关**，
  这正是 TaskAligned 要治的 task misalignment。

In [ ]:
def bbox_iou(a, b):
    '''a:(M,4) b:(N,4) xyxy -> (M,N) 的 IoU 矩阵。'''
    area_a = (a[:, 2] - a[:, 0]).clip(0) * (a[:, 3] - a[:, 1]).clip(0)
    area_b = (b[:, 2] - b[:, 0]).clip(0) * (b[:, 3] - b[:, 1]).clip(0)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clip(0)
    inter = wh[..., 0] * wh[..., 1]
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-12)

def points_in_boxes(points, boxes):
    '''(G,N) 布尔：格点中心是否严格落在框内。'''
    cx, cy = points[:, 0], points[:, 1]
    return ((cx[None] > boxes[:, 0:1]) & (cx[None] < boxes[:, 2:3]) &
            (cy[None] > boxes[:, 1:2]) & (cy[None] < boxes[:, 3:4]))

# —— IoU 自校验 ——
_t = np.array([[0., 0., 10., 10.], [5., 5., 15., 15.], [100., 100., 110., 110.]])
_m = bbox_iou(_t, _t)
assert np.allclose(np.diag(_m), 1.0)
assert _m[0, 2] == 0.0
assert abs(_m[0, 1] - 25 / 175) < 1e-9        # 交 5x5=25，并 100+100-25=175
print(f'IoU 自校验通过：半重叠框 IoU = {_m[0,1]:.4f} = 25/175')

# —— IoU 对位移的敏感性：小目标为什么这么难（C57 会展开）——
print()
print(f'{"框边长":>8s} {"偏移1px":>9s} {"偏移2px":>9s} {"偏移4px":>9s}')
for L in [8, 16, 32, 64]:
    row = []
    for d in [1, 2, 4]:
        a = np.array([[0., 0., L, L]]); b = np.array([[d, d, L + d, L + d]])
        row.append(bbox_iou(a, b)[0, 0])
    print(f'{L:>6d}px {row[0]:>9.3f} {row[1]:>9.3f} {row[2]:>9.3f}')
assert bbox_iou(np.array([[0., 0., 8., 8.]]), np.array([[2., 2., 10., 10.]]))[0, 0] < 0.40
print('⚠️  8x8 的框偏移 2px，IoU 就掉到 0.39；64x64 的框偏 2px 还有 0.88。')
print('    同一个 IoU 阈值对不同尺度是**极不公平**的 —— 这是本模块所有问题的根。')

In [ ]:
def synth_detector(points, gt, gt_cls, n_cls=N_CLS, seed=7):
    '''合成一个「训练到一半」的检测器输出：(cls_score[N,C], pred_box[N,4])。'''
    rg = np.random.default_rng(seed)
    N = len(points)
    ctr = (gt[:, :2] + gt[:, 2:]) / 2
    sz = np.sqrt((gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1]))
    d = np.linalg.norm(points[:, None, :] - ctr[None, :, :], axis=2) / sz[None, :]
    near, dmin = d.argmin(1), d.min(1)
    q = np.exp(-(dmin / 0.7) ** 2)                # 预测质量 in (0,1]：离中心越近越准
    g = gt[near]
    gcx, gcy = (g[:, 0] + g[:, 2]) / 2, (g[:, 1] + g[:, 3]) / 2
    gw, gh = g[:, 2] - g[:, 0], g[:, 3] - g[:, 1]
    j = 1 - q                                     # 抖动强度
    nz = rg.normal(size=(N, 4))
    pcx = gcx + nz[:, 0] * 0.45 * gw * j
    pcy = gcy + nz[:, 1] * 0.45 * gh * j
    pw = gw * np.exp(nz[:, 2] * 0.40 * j)         # 用 log 空间抖动 -> 框永远合法
    ph = gh * np.exp(nz[:, 3] * 0.40 * j)
    box = np.stack([pcx - pw / 2, pcy - ph / 2, pcx + pw / 2, pcy + ph / 2], 1)
    logit = rg.normal(-3.6, 0.6, size=(N, n_cls))                 # 背景基线 sigmoid≈0.03
    logit[np.arange(N), gt_cls[near]] += 6.6 * q + rg.normal(0, 0.9, N)   # ← 解耦噪声
    return 1 / (1 + np.exp(-logit)), box

CLS_PRED, BOX_PRED = synth_detector(POINTS, GT, GT_CLS)
IOU_PRED = bbox_iou(GT, BOX_PRED)          # (G,N) 预测框 vs GT
IN_GT = points_in_boxes(POINTS, GT)

print(f'{"GT":>3s} {"尺寸":>7s} {"框内格点数":>11s} {"(s8/s16/s32)":>14s} {"最好预测IoU":>12s} {"corr(分数,IoU)":>15s}')
for g in range(len(GT)):
    per_lv = [int((IN_GT[g] & (PT_L == l)).sum()) for l in range(3)]
    near_mask = IN_GT[g] | (bbox_iou(GT[g:g+1], ANCHORS)[0] > 0.05)
    c = np.corrcoef(CLS_PRED[near_mask, GT_CLS[g]], IOU_PRED[g, near_mask])[0, 1]
    print(f'{g:>3d} {GT_WH[g,0]:>4.0f}px {int(IN_GT[g].sum()):>11d} '
          f'{str(per_lv):>14s} {IOU_PRED[g].max():>12.3f} {c:>15.3f}')

assert IOU_PRED.max(1).min() > 0.80, '每个 GT 附近都应有一个近乎完美的预测'
assert IN_GT.sum(1)[0] == 2, '16x16 的 GT 全图只有 2 个格点中心落在框内'
print()
print('⚠️  第 3 列是本 notebook 最重要的一个数字：')
print('    **16x16 的标志，全图 8400 个格点里只有 2 个中心落在框内。**')
print('    正样本的绝对上限被 stride 卡死了 —— 分配算法再聪明也变不出格点。')
print('⚠️  最后一列 0.6~0.8：分类分数与定位质量**相关但不等同** -> task misalignment。')

## 3 · 静态 MaxIoU 分配：先算清「哪些目标根本没资格」

RetinaNet 式规则：IoU ≥ 0.5 为正、< 0.4 为负、中间忽略，
外加「每个 GT 强制认领它 IoU 最高的 anchor」这条兜底。

**先做一件比跑算法更有价值的事：算理论上界。**

In [ ]:
IOU_ANC = bbox_iou(GT, ANCHORS)     # (G,N)：anchor 与 GT 的几何 IoU（与预测无关）

print('每个 GT 在各层 anchor 上的**理论最大 IoU** = min(A_gt,A_anc)/max(A_gt,A_anc)')
print(f'{"GT":>3s} {"尺寸":>7s} {"s8(32px)":>10s} {"s16(64px)":>10s} {"s32(128px)":>11s} {"实测max":>9s}')
for g in range(len(GT)):
    a_gt = GT_WH[g].prod()
    ub = [min(a_gt, (ANCHOR_SCALE * s) ** 2) / max(a_gt, (ANCHOR_SCALE * s) ** 2) for s in STRIDES]
    mark = '  ← 天花板 < 0.5' if max(ub) < 0.5 else ''
    print(f'{g:>3d} {GT_WH[g,0]:>4.0f}px {ub[0]:>10.3f} {ub[1]:>10.3f} {ub[2]:>11.3f} '
          f'{IOU_ANC[g].max():>9.3f}{mark}')

# 16px / 18px 的 GT：无论哪一层，几何 IoU 上界都低于 0.5
assert IOU_ANC[0].max() < 0.5 and IOU_ANC[1].max() < 0.5
assert IOU_ANC[2].max() >= 0.5
print()
print('⚠️  16px 与 18px 的标志：**在任何一层上都不可能达到 IoU 0.5**。')
print('    不是「难匹配」，是数学上不可能 —— 固定阈值对它们是一道硬门。')

In [ ]:
def maxiou_assign(iou_gt_anchor, pos_thr=0.5, neg_thr=0.4, force_best=True):
    '''RetinaNet 式静态分配。返回 assign[N]（-1=负样本，>=0 是 GT 下标）与 ignore 掩码。'''
    G, N = iou_gt_anchor.shape
    best_gt, best_iou = iou_gt_anchor.argmax(0), iou_gt_anchor.max(0)
    assign = np.full(N, -1)
    hit = best_iou >= pos_thr
    assign[hit] = best_gt[hit]
    ignore = (best_iou >= neg_thr) & (best_iou < pos_thr)
    if force_best:                                # 低质量兜底：每个 GT 至少领一个 anchor
        for g in range(G):
            assign[iou_gt_anchor[g].argmax()] = g
    return assign, ignore

A_MAXIOU, IGNORE = maxiou_assign(IOU_ANC)
n_thr = int((IOU_ANC.max(0) >= 0.5).sum())
print(f'靠阈值 0.5 拿到的正样本: {n_thr} 个')
print(f'加上「强制认领」兜底后:   {int((A_MAXIOU >= 0).sum())} 个')
print(f'忽略样本: {int(IGNORE.sum())} 个,  负样本: {int((A_MAXIOU < 0).sum() - IGNORE.sum())} 个')
print()
print(f'{"GT":>3s} {"尺寸":>7s} {"正样本数":>9s} {"来源":>28s}')
for g in range(len(GT)):
    n = int((A_MAXIOU == g).sum())
    src = '全部来自兜底（阈值一个没中）' if (IOU_ANC[g] >= 0.5).sum() == 0 else '阈值命中'
    print(f'{g:>3d} {GT_WH[g,0]:>4.0f}px {n:>9d} {src:>28s}')

assert (A_MAXIOU >= 0).sum() <= 8, '静态分配在小目标场景下正样本极度稀缺'
assert (A_MAXIOU == 0).sum() == 1 and (A_MAXIOU == 1).sum() == 1
print()
print('⚠️  8400 个格点，只有 6 个正样本，正负比 1:1400。')
print('    16px 与 18px 的两个标志，唯一的正样本**全靠兜底规则硬塞**——')
print(f'    塞进来的 anchor IoU 只有 {IOU_ANC[0].max():.2f} / {IOU_ANC[1].max():.2f}：')
print('    这是**噪声监督**，不是好监督。而 48px / 96px 各只拿到 1 个阈值命中的正样本。')

## 4 · ATSS：让阈值自己从候选集的统计量里长出来

四步：每层取最近 k=9 个 anchor 当候选 → 算它们与 GT 的 IoU →
**阈值 = mean + std** → 过阈值且中心在框内的为正样本。

注意 `std` 用 **ddof=1**（无偏），与 PyTorch / mmdet 的默认一致。

In [ ]:
def atss_assign(points, pt_level, anchors, gt, topk=9, n_levels=3):
    '''ATSS。返回 assign[N], 每个 GT 的自适应阈值 thr[G], 候选掩码 cand[G,N]。'''
    G, N = len(gt), len(anchors)
    gctr = (gt[:, :2] + gt[:, 2:]) / 2
    dist = np.linalg.norm(points[:, None, :] - gctr[None, :, :], axis=2)   # (N,G)
    iou_anc = bbox_iou(gt, anchors)                                        # (G,N)

    cand = np.zeros((G, N), bool)
    for g in range(G):
        for l in range(n_levels):                     # ← **每层各取 topk**，不是全局 topk
            idx = np.where(pt_level == l)[0]
            cand[g, idx[np.argsort(dist[idx, g])[:min(topk, len(idx))]]] = True

    inside = points_in_boxes(points, gt)
    thr = np.zeros(G)
    pos = np.zeros((G, N), bool)
    for g in range(G):
        v = iou_anc[g, cand[g]]
        thr[g] = v.mean() + v.std(ddof=1)             # ← 自适应阈值
        pos[g] = cand[g] & (iou_anc[g] >= thr[g]) & inside[g]

    assign = np.full(N, -1)                            # 冲突 -> IoU 最大的 GT 胜
    any_pos = pos.any(0)
    assign[any_pos] = np.where(pos, iou_anc, -1.0)[:, any_pos].argmax(0)
    return assign, thr, cand

A_ATSS, ATSS_THR, ATSS_CAND = atss_assign(POINTS, PT_L, ANCHORS, GT)

print(f'{"GT":>3s} {"尺寸":>7s} {"候选数":>7s} {"候选IoU均值":>12s} {"标准差":>8s} '
      f'{"自适应阈值":>11s} {"正样本":>7s}')
iou_anc = bbox_iou(GT, ANCHORS)
for g in range(len(GT)):
    v = iou_anc[g, ATSS_CAND[g]]
    print(f'{g:>3d} {GT_WH[g,0]:>4.0f}px {int(ATSS_CAND[g].sum()):>7d} {v.mean():>12.3f} '
          f'{v.std(ddof=1):>8.3f} {ATSS_THR[g]:>11.3f} {int((A_ATSS == g).sum()):>7d}')

assert ATSS_THR[0] < 0.30, '16px 的 GT 阈值应远低于 0.5'
assert ATSS_THR[3] > 0.45, '48px 的 GT 阈值应接近 0.5'
assert (A_ATSS >= 0).sum() > (A_MAXIOU >= 0).sum(), 'ATSS 应比 MaxIoU 给出更多正样本'
print()
print(f'⚠️  自适应阈值从 {ATSS_THR.min():.3f}（16px）到 {ATSS_THR.max():.3f}（48px）——')
print('    **固定 0.5 相当于对小目标单方面提高了 2.3 倍的门槛。**')
print('✅ ATSS 的全部魔法就是 mean+std：std 大说明某一层特别匹配 -> 抬阈值只选那层；')
print('   std 小说明各层半斤八两 -> 降阈值多选几个。选层与选数量被同一个公式解决了。')

## 5 · SimOTA：代价矩阵 → dynamic-k → 去冲突

这是本模块的技术核心，也是 YOLOX 相对 YOLOv5 的最大增量。四步逐个实现。

In [ ]:
def simota_assign(points, pt_stride, gt, gt_cls, cls_pred, box_pred,
                  center_radius=2.5, n_candidate_k=10, lam_reg=3.0,
                  soft_label=False, big_penalty=1e5):
    '''SimOTA（YOLOX）。返回 assign[N] 与一份完整的中间结果，便于逐步检查。'''
    G, N, C = len(gt), len(points), cls_pred.shape[1]
    cx, cy = points[:, 0], points[:, 1]

    # ---- ① 几何先验：候选池 = in_box OR in_center；先验掩码 = in_box AND in_center ----
    in_box = points_in_boxes(points, gt)
    gcx, gcy = (gt[:, 0:1] + gt[:, 2:3]) / 2, (gt[:, 1:2] + gt[:, 3:4]) / 2
    r = center_radius * pt_stride[None, :]                 # 半径随层的 stride 放大
    in_ctr = ((cx[None] > gcx - r) & (cx[None] < gcx + r) &
              (cy[None] > gcy - r) & (cy[None] < gcy + r))
    fg = (in_box | in_ctr).any(0)
    idx = np.where(fg)[0]
    M = len(idx)
    both = in_box[:, idx] & in_ctr[:, idx]                 # (G,M)

    # ---- ② 代价矩阵 ----
    ious = bbox_iou(gt, box_pred[idx])                     # (G,M)
    p = np.sqrt(np.clip(cls_pred[idx], 1e-8, 1 - 1e-8))    # YOLOX 的 sqrt（cls×obj 的几何均值）
    y = np.zeros((G, M, C))
    y[np.arange(G), :, gt_cls] = 1.0
    if soft_label:                                         # RTMDet 路线：目标值 = IoU
        y = y * ious[:, :, None]
    cls_cost = -(y * np.log(p)[None] + (1 - y) * np.log(1 - p)[None]).sum(-1)
    reg_cost = -np.log(np.clip(ious, 1e-8, None))
    cost = cls_cost + lam_reg * reg_cost + big_penalty * (~both)

    # ---- ③ dynamic-k：配额由预测质量自己决定 ----
    nk = min(n_candidate_k, M)
    dyn_k = np.clip(np.sort(ious, 1)[:, -nk:].sum(1).astype(int), 1, None)

    matching = np.zeros((G, M), bool)
    for g in range(G):
        matching[g, np.argsort(cost[g])[:dyn_k[g]]] = True

    # ---- ④ 去冲突：一个格点被多个 GT 选中 -> 归给代价最小的 GT ----
    multi = matching.sum(0) > 1
    n_conflict = int(multi.sum())
    if multi.any():
        win = np.where(matching, cost, np.inf)[:, multi].argmin(0)
        matching[:, multi] = False
        matching[win, np.where(multi)[0]] = True

    assign = np.full(N, -1)
    has = matching.any(0)
    assign[idx[has]] = matching[:, has].argmax(0)
    return assign, dict(fg_idx=idx, cost=cost, ious=ious, both=both,
                        dyn_k=dyn_k, n_conflict=n_conflict, matching=matching)

A_SIMOTA, D = simota_assign(POINTS, PT_S, GT, GT_CLS, CLS_PRED, BOX_PRED)

print(f'① 候选池：8400 个格点 -> {len(D["fg_idx"])} 个（in_box OR in_center, r=2.5·stride）')
print(f'② 代价矩阵 shape = {D["cost"].shape}')
print(f'④ 去冲突：有 {D["n_conflict"]} 个格点被多个 GT 同时选中')
print()
print(f'{"GT":>3s} {"尺寸":>7s} {"AND先验内":>10s} {"dynamic-k":>10s} {"最终正样本":>11s} {"其中在框外":>11s}')
for g in range(len(GT)):
    sel = np.where(D['matching'][g])[0]
    out = int((~D['both'][g, sel]).sum())
    print(f'{g:>3d} {GT_WH[g,0]:>4.0f}px {int(D["both"][g].sum()):>10d} {D["dyn_k"][g]:>10d} '
          f'{int((A_SIMOTA == g).sum()):>11d} {out:>11d}')

assert len(D['fg_idx']) < 600, '几何先验应把 8400 砍到几百'
assert D['dyn_k'][4] >= D['dyn_k'][0], '大目标/预测好的目标应拿到更大的配额'
assert D['dyn_k'].min() >= 1, 'dynamic-k 至少为 1'
assert (A_SIMOTA >= 0).sum() > (A_ATSS >= 0).sum()
print()
print('⚠️  看最后一列：16px 的 GT dynamic-k=6，但满足 AND 先验的格点只有 2 个 ——')
print('    于是另外 4 个正样本**落在了 GT 框外**。这不是 bug，是 YOLOX 原版行为：')
print('    那个 100000 只影响排序、不是硬约束。dynamic-k 会把它顶穿。')
print('✅ 这正是 RTMDet 用**连续衰减的软中心先验**替换 0/100000 悬崖的动机之一。')

In [ ]:
# —— dynamic-k 到底在干什么：把「配额」画出来 ——
print('dynamic-k = clamp( floor( 该 GT 周围 top-10 预测框 IoU 之和 ), min=1 )')
print()
print(f'{"GT":>3s} {"尺寸":>7s} {"top-10 预测 IoU":>44s} {"和":>7s} {"k":>4s}')
for g in range(len(GT)):
    top = np.sort(D['ious'][g])[-10:][::-1]
    print(f'{g:>3d} {GT_WH[g,0]:>4.0f}px {" ".join(f"{v:.2f}" for v in top):>44s} '
          f'{top.sum():>7.2f} {D["dyn_k"][g]:>4d}')

# 训练早期（预测很差）时 dynamic-k 会自动退化成一对一
bad_box = BOX_PRED + np.random.default_rng(0).normal(0, 60, BOX_PRED.shape)
_, D_bad = simota_assign(POINTS, PT_S, GT, GT_CLS, CLS_PRED * 0.1, bad_box)
print()
print(f'训练初期（预测框加 60px 噪声、分数压到 1/10）的 dynamic-k: {D_bad["dyn_k"]}')
print(f'训练中期（本例的正常预测）的 dynamic-k:                   {D["dyn_k"]}')
assert D_bad['dyn_k'].sum() < D['dyn_k'].sum(), '预测越差，配额应越小'
print()
print('✅ 这是 dynamic-k 最漂亮的性质：**它随训练进度自动调节监督密度**。')
print('   早期预测烂 -> k 接近 1，等价于一对一，不灌噪声；')
print('   后期预测好 -> k 涨到 8~10，监督变密，收敛加速。无需任何手工 schedule。')

In [ ]:
# —— 去冲突：用一个手算得出来的小矩阵把规则钉死 ——
toy_cost = np.array([
    [0.5, 0.8, 1.0, 3.0, 4.0, 5.0],     # GT0 对 6 个候选格点的代价
    [4.0, 3.0, 1.2, 0.9, 0.6, 5.0],     # GT1
])
toy_k = np.array([3, 3])

def take_topk(cost, k):
    m = np.zeros(cost.shape, bool)
    for g in range(len(cost)):
        m[g, np.argsort(cost[g])[:k[g]]] = True
    return m

def resolve(matching, cost):
    multi = matching.sum(0) > 1
    n = int(multi.sum())
    if multi.any():
        win = np.where(matching, cost, np.inf)[:, multi].argmin(0)
        matching[:, multi] = False
        matching[win, np.where(multi)[0]] = True
    return matching, n

m0 = take_topk(toy_cost, toy_k)
print('去冲突前：')
print('  GT0 选中格点', np.where(m0[0])[0], ' GT1 选中格点', np.where(m0[1])[0])
print('  格点 2 被两个 GT 同时选中：cost[0,2]=1.0 < cost[1,2]=1.2 -> 判给 GT0')
m1, nconf = resolve(m0.copy(), toy_cost)
print('去冲突后：')
print('  GT0 ->', np.where(m1[0])[0], ' GT1 ->', np.where(m1[1])[0], f'  (仲裁了 {nconf} 个格点)')

assert nconf == 1
assert list(np.where(m1[0])[0]) == [0, 1, 2]
assert list(np.where(m1[1])[0]) == [3, 4]
assert m1.sum(0).max() == 1, '去冲突后每个格点最多属于一个 GT'
print()
print('⚠️  注意 GT1 的正样本从 3 个变成了 2 个 —— **被抢走的名额不补**。')
print('    在 TSR 的「主牌 + 辅助牌」构型里，本来只有 2 个格点的辅助牌')
print('    被主牌抢走 1 个就只剩 1 个正样本 -> 线上表现为「辅助牌时有时无」。')
print('✅ 完整 OTA 用约束 Σ_i π[i,g] = k_g 保证每个 GT 拿满配额，代价是 Sinkhorn 迭代。')

## 6 · TaskAligned：$t = s^{\alpha} \cdot u^{\beta}$

TOOD / YOLOv8 路线。同一个对齐度量既用来**选样本**、也用来当**软标签**。
这里只实现选择部分（软标签部分见模块 03）。

In [ ]:
def task_aligned_assign(points, gt, gt_cls, cls_pred, box_pred,
                        topk=13, alpha=1.0, beta=6.0):
    '''TaskAligned 分配。返回 assign[N], 对齐度矩阵 t[G,N], 冲突数。'''
    G, N = len(gt), len(points)
    inside = points_in_boxes(points, gt)
    ious = bbox_iou(gt, box_pred).clip(0)                 # (G,N)
    s = cls_pred[:, gt_cls].T                             # (G,N) 该 GT 类别上的分数
    t = (s ** alpha) * (ious ** beta)                     # ← 对齐度
    t_masked = np.where(inside, t, 0.0)

    match = np.zeros((G, N), bool)
    for g in range(G):
        k = min(topk, int(inside[g].sum()))
        if k == 0:                                        # 兜底：没有格点落在框内
            d = np.linalg.norm(points - (gt[g, :2] + gt[g, 2:]) / 2, axis=1)
            match[g, d.argmin()] = True
            continue
        sel = np.argsort(-t_masked[g])[:k]
        match[g, sel[t_masked[g, sel] > 0]] = True

    multi = match.sum(0) > 1                              # 去冲突：YOLOv8 用「IoU 最大者胜」
    n_conflict = int(multi.sum())
    if multi.any():
        win = np.where(match, ious, -1.0)[:, multi].argmax(0)
        match[:, multi] = False
        match[win, np.where(multi)[0]] = True

    assign = np.full(N, -1)
    any_m = match.any(0)
    assign[any_m] = match[:, any_m].argmax(0)
    return assign, t, n_conflict

A_TAL, T_ALIGN, TAL_CONF = task_aligned_assign(POINTS, GT, GT_CLS, CLS_PRED, BOX_PRED)

print(f'{"GT":>3s} {"尺寸":>7s} {"框内格点":>9s} {"topk上限":>9s} {"实得正样本":>11s} '
      f'{"正样本平均IoU":>14s}')
for g in range(len(GT)):
    m = A_TAL == g
    print(f'{g:>3d} {GT_WH[g,0]:>4.0f}px {int(IN_GT[g].sum()):>9d} {min(13, int(IN_GT[g].sum())):>9d} '
          f'{int(m.sum()):>11d} {IOU_PRED[g, m].mean():>14.3f}')

# beta 远大于 alpha：定位质量压倒分类分数
print()
print(f'{"":>16s} {"s=0.9,u=0.9":>13s} {"s=0.9,u=0.8":>13s} {"s=0.8,u=0.9":>13s}')
for a, b in [(1.0, 6.0), (0.5, 6.0)]:
    vals = [(0.9 ** a) * (0.9 ** b), (0.9 ** a) * (0.8 ** b), (0.8 ** a) * (0.9 ** b)]
    print(f'alpha={a}, beta={b}  {vals[0]:>13.4f} {vals[1]:>13.4f} {vals[2]:>13.4f}')

t_iou_drop = (0.9 ** 1.0) * (0.8 ** 6.0)
t_cls_drop = (0.8 ** 1.0) * (0.9 ** 6.0)
assert t_cls_drop > t_iou_drop * 1.5, 'IoU 掉 0.1 的惩罚应远重于分数掉 0.1'
assert (A_TAL == 0).sum() <= 2, '16px 的 GT 框内只有 2 个格点，TaskAligned 最多给 2 个'
print()
print(f'⚠️  IoU 从 0.9->0.8，对齐度掉 {(1 - t_iou_drop / ((0.9**1)*(0.9**6))) * 100:.0f}%；')
print(f'    分数从 0.9->0.8，对齐度只掉 {(1 - t_cls_drop / ((0.9**1)*(0.9**6))) * 100:.0f}%。')
print('✅ beta >> alpha 是刻意的：**TaskAligned 主要按「框准不准」选样本**，')
print('   分类分数只是温和的修正项。这也是它对 misalignment 有效的原因。')

## 7 · 四种分配的正样本集合差异：一张表说完

同一组数据、同一个合成检测器，四种分配给出的正样本集合到底差多少。

In [ ]:
METHODS = {'MaxIoU@0.5': A_MAXIOU, 'ATSS': A_ATSS,
           'SimOTA': A_SIMOTA, 'TaskAligned': A_TAL}

print(f'{"方法":<13s} {"正样本":>7s} {"每GT分布":>22s} {"层分布(s8/s16/s32)":>21s} '
      f'{"平均IoU":>9s} {"平均分数":>9s}')
stats = {}
for name, a in METHODS.items():
    pos = a >= 0
    idxs = np.where(pos)[0]
    per_gt = [int((a == g).sum()) for g in range(len(GT))]
    per_lv = [int((pos & (PT_L == l)).sum()) for l in range(3)]
    m_iou = float(IOU_PRED[a[pos], idxs].mean())
    m_scr = float(CLS_PRED[idxs, GT_CLS[a[pos]]].mean())
    stats[name] = dict(n=int(pos.sum()), per_gt=per_gt, per_lv=per_lv,
                       iou=m_iou, score=m_scr)
    print(f'{name:<13s} {pos.sum():>7d} {str(per_gt):>22s} {str(per_lv):>21s} '
          f'{m_iou:>9.3f} {m_scr:>9.3f}')

# 正样本「质量」：预测感知的分配器挑出的样本定位更准
assert stats['MaxIoU@0.5']['iou'] < stats['TaskAligned']['iou']
assert stats['MaxIoU@0.5']['score'] < stats['TaskAligned']['score']
assert stats['MaxIoU@0.5']['n'] < stats['ATSS']['n'] < stats['SimOTA']['n']
print()
print('✅ 平均 IoU 单调上升 0.808 -> 0.848 -> 0.852 -> 0.885：')
print('   **越是「看着预测选样本」的分配器，挑出的正样本定位质量越高。**')
print('   这不是废话 —— 它意味着回归损失看到的是更容易学的样本，梯度信噪比更好。')

In [ ]:
def jaccard(a, b):
    sa, sb = set(np.where(a >= 0)[0]), set(np.where(b >= 0)[0])
    return len(sa & sb) / max(len(sa | sb), 1)

names = list(METHODS)
print('正样本集合的 Jaccard 重叠：')
print(f'{"":<13s}' + ''.join(f'{n:>13s}' for n in names))
J = np.zeros((4, 4))
for i, ni in enumerate(names):
    row = ''
    for j, nj in enumerate(names):
        J[i, j] = jaccard(METHODS[ni], METHODS[nj])
        row += f'{J[i,j]:>13.3f}'
    print(f'{ni:<13s}{row}')

assert np.allclose(np.diag(J), 1.0)
assert J[2, 3] > J[0, 1], 'SimOTA 与 TaskAligned 更像（都看预测），MaxIoU 与 ATSS 更不像'
assert J[0, 2] < 0.25, 'MaxIoU 与 SimOTA 的正样本集合几乎不重叠'
print()
print(f'⚠️  MaxIoU 与 SimOTA 的重叠只有 {J[0,2]:.1%} —— **它们在训练完全不同的东西**。')
print(f'✅ SimOTA 与 TaskAligned 重叠 {J[2,3]:.1%}：都以「预测质量」为主要依据，殊途同归。')
print()
print('把「选择依据」竖着读，就是标签分配十年演进的全部内容：')
print('  几何 IoU -> 几何 IoU 的自适应统计 -> 分类+定位联合代价 -> 分类×定位的乘性对齐度')
print('  信息源从「框和框的关系」一路搬到了「模型当前的预测本身」。')

## ✏️ 练习 1：静态阈值的「可匹配尺寸区间」

推导并实现 `matchable_size_range(anchor_side, thr)`：在「同宽高比、同心」的理想情况下，
GT 边长 $L$ 与 anchor 边长 $s$ 的 IoU 上界是 $\min(L^2,s^2)/\max(L^2,s^2)$。
求使 IoU $\ge \tau$ 的 $L$ 区间，返回 `(lo, hi)`。

再实现 `is_matchable(gt_side, strides, anchor_scale, thr)`：判断某个尺寸的 GT
在**任意一层**上能否被匹配上。

In [ ]:
def matchable_size_range(anchor_side, thr=0.5):
    # TODO: 解 min(L^2,s^2)/max(L^2,s^2) >= thr，返回 (lo, hi)
    raise NotImplementedError

def is_matchable(gt_side, strides=STRIDES, anchor_scale=ANCHOR_SCALE, thr=0.5):
    # TODO: 任意一层能匹配上就返回 True
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
lo, hi = matchable_size_range(32.0, 0.5)
assert abs(lo - 32 / np.sqrt(2)) < 1e-9 and abs(hi - 32 * np.sqrt(2)) < 1e-9, (lo, hi)
assert abs(lo - 22.627) < 1e-3 and abs(hi - 45.255) < 1e-3
lo9, hi9 = matchable_size_range(32.0, 0.9)
assert hi9 - lo9 < hi - lo, '阈值越高，可匹配区间越窄'
assert matchable_size_range(64.0, 0.5)[0] == 2 * lo, '区间随 anchor 边长线性缩放'

assert not is_matchable(16.0) and not is_matchable(18.0), '16/18px 在任何一层都匹配不上'
assert is_matchable(24.0) and is_matchable(48.0) and is_matchable(96.0)
assert not is_matchable(300.0), '超大目标同样会掉出区间'

print(f'{"anchor边长":>10s} {"可匹配GT尺寸区间(τ=0.5)":>26s}')
for s in STRIDES:
    a = ANCHOR_SCALE * s
    r = matchable_size_range(a, 0.5)
    print(f'{a:>9.0f}px {f"[{r[0]:.1f}, {r[1]:.1f}]":>26s}')
print()
print(f'{"GT尺寸":>8s} {"能否被 IoU>=0.5 匹配":>22s}')
for L in [9, 16, 18, 24, 40, 48, 96, 200, 300]:
    print(f'{L:>6d}px {("✅ 可以" if is_matchable(float(L)) else "❌ 不可能"):>22s}')
print()
print('✅ 练习 1 通过：三档 anchor 覆盖 [22.6, 181]，**9~22px 的目标是一片盲区**。')
print('   TSR 里 80m 外的限速牌正好 9px —— 这就是为什么必须加 P2 层或提分辨率。')

## ✏️ 练习 2：从零实现 dynamic-k 估计器

实现 `dynamic_k(ious, n_candidate_k=10, k_min=1)`：
`ious` 是 `(G, M)` 的「GT × 候选格点」IoU 矩阵，
返回长度 G 的整数配额数组 —— **每个 GT 取自己 top-`n_candidate_k` 个 IoU 求和后向下取整，再 clamp 到 ≥ k_min**。

不许调用上面 `simota_assign` 里的实现，写完与它对拍。

In [ ]:
def dynamic_k(ious, n_candidate_k=10, k_min=1):
    # TODO: 返回 shape (G,) 的 int 数组
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
toy = np.array([
    [0.9, 0.85, 0.8, 0.1, 0.05],      # top-3 和 = 2.55 -> k=2
    [0.3, 0.2,  0.1, 0.05, 0.0],      # top-3 和 = 0.60 -> floor=0 -> clamp 到 1
])
k = dynamic_k(toy, n_candidate_k=3)
assert k.dtype.kind == 'i', '应返回整数数组'
assert list(k) == [2, 1], k
assert list(dynamic_k(toy, n_candidate_k=5)) == [2, 1]
assert list(dynamic_k(np.ones((2, 20)), n_candidate_k=10)) == [10, 10], '全 1 时 k = n_candidate_k'
assert list(dynamic_k(np.zeros((3, 8)), n_candidate_k=4)) == [1, 1, 1], '全 0 时 clamp 到 1'
assert list(dynamic_k(toy, n_candidate_k=3, k_min=2)) == [2, 2]

# 与 worked cell 的 SimOTA 对拍
k_ref = D['dyn_k']
k_mine = dynamic_k(D['ious'], n_candidate_k=10)
assert np.array_equal(k_mine, k_ref), (k_mine, k_ref)
print('与 worked SimOTA 对拍一致:', k_mine)
print()
print(f'{"GT":>3s} {"尺寸":>7s} {"k(正常预测)":>12s} {"k(预测退化50%)":>16s}')
for g in range(len(GT)):
    kd = dynamic_k(D['ious'] * 0.5, 10)[g]
    print(f'{g:>3d} {GT_WH[g,0]:>4.0f}px {k_mine[g]:>12d} {kd:>16d}')
assert dynamic_k(D['ious'] * 0.5, 10).sum() < k_mine.sum()
print()
print('✅ 练习 2 通过：dynamic-k 把「每个 GT 分几个正样本」交给了模型自己。')

## ✏️ 练习 3：分配诊断报告

线上调检测器时，最有用的不是 AP，而是**分配本身的统计**。实现
`assign_report(assign, gt_wh, pt_level, n_levels=3)`，返回字典：

- `n_pos`：正样本总数
- `per_gt`：每个 GT 的正样本数（list）
- `per_level`：每层的正样本数（list）
- `starved`：正样本数 ≤ 1 的 GT 下标（list）——**这是最该报警的指标**
- `imbalance`：`max(per_gt) / max(min(per_gt), 1)`，尺度公平性的粗略度量

In [ ]:
def assign_report(assign, gt_wh, pt_level, n_levels=3):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
r = assign_report(A_MAXIOU, GT_WH, PT_L)
assert r['n_pos'] == int((A_MAXIOU >= 0).sum())
assert sum(r['per_gt']) == r['n_pos'] and sum(r['per_level']) == r['n_pos']
assert len(r['per_level']) == 3
assert 0 in r['starved'] and 1 in r['starved'], 'MaxIoU 下 16/18px 的 GT 都饿着'
assert r['imbalance'] == max(r['per_gt']) / max(min(r['per_gt']), 1)

r2 = assign_report(A_SIMOTA, GT_WH, PT_L)
assert len(r2['starved']) == 0, 'SimOTA 下没有 GT 只拿到 <=1 个正样本'
assert r2['imbalance'] < r['imbalance'], 'SimOTA 的尺度公平性优于 MaxIoU'

print(f'{"方法":<13s} {"正样本":>7s} {"饿着的GT":>22s} {"不均衡度":>9s} {"层分布":>18s}')
for name, a in METHODS.items():
    rr = assign_report(a, GT_WH, PT_L)
    starve = str([f'#{i}({GT_WH[i,0]:.0f}px)' for i in rr['starved']]) if rr['starved'] else '无'
    print(f'{name:<13s} {rr["n_pos"]:>7d} {starve:>22s} {rr["imbalance"]:>9.2f} '
          f'{str(rr["per_level"]):>18s}')
print()
print('✅ 练习 3 通过：**把这份报告打进训练日志**，比等 mAP 出来再回头猜有效得多。')
print('   线上真实用法：某个类别 AP 突然掉 -> 先看它的 GT 是不是在 starved 名单里。')

## ✏️ 练习 4：RTMDet 的软中心先验

SimOTA 的中心先验是 0/100000 的悬崖，会被 dynamic-k 顶穿。
RTMDet 换成连续衰减的 $10^{\,d/s - R}$（$d$ = 格点到 GT 中心的像素距离，
$s$ = 该格点的 stride，$R$ = `soft_center_radius`，默认 3.0）。

实现 `soft_center_prior(points, pt_stride, gt, radius=3.0)`，返回 `(G, N)` 的代价矩阵。

In [ ]:
def soft_center_prior(points, pt_stride, gt, radius=3.0):
    # TODO: 10 ** (归一化距离 - radius)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
sp = soft_center_prior(POINTS, PT_S, GT)
assert sp.shape == (len(GT), len(POINTS))

# 关键性质：d=0 -> 1e-3；d=radius*stride -> 1；d=5*stride -> 100
one_pt = np.array([[100.0, 100.0]]); one_s = np.array([8.0])
g_at = np.array([[92., 92., 108., 108.]])              # 中心正好是 (100,100)
assert abs(soft_center_prior(one_pt, one_s, g_at)[0, 0] - 1e-3) < 1e-9
g_off = np.array([[68., 92., 84., 108.]])              # 中心 (76,100)，距离 24 = 3*stride
assert abs(soft_center_prior(one_pt, one_s, g_off)[0, 0] - 1.0) < 1e-9
g_far = np.array([[52., 92., 68., 108.]])              # 中心 (60,100)，距离 40 = 5*stride
assert abs(soft_center_prior(one_pt, one_s, g_far)[0, 0] - 100.0) < 1e-6

print(f'{"距离/stride":>12s} {"软中心先验代价":>16s} {"对比 SimOTA 硬先验":>22s}')
soft_curve = np.array([10.0 ** (dn - 3.0) for dn in range(7)])
for dn in range(7):
    hard = '0（框内+中心内）' if dn <= 2.5 else '100000（悬崖）'
    print(f'{dn:>12d} {soft_curve[dn]:>16.4f} {hard:>22s}')
assert np.all(np.diff(soft_curve) > 0), '软先验必须随距离严格单调增'

# 把软先验接进 SimOTA 的代价：拆掉 0/100000 悬崖，换成斜坡
sp_fg = sp[:, D['fg_idx']]
cost_soft = D['cost'] - 1e5 * (~D['both']) + sp_fg
k = dynamic_k(D['ious'], 10)
match_soft = np.zeros_like(D['matching'])
for g in range(len(GT)):
    match_soft[g, np.argsort(cost_soft[g])[:k[g]]] = True

gctr = (GT[:, :2] + GT[:, 2:]) / 2
dnorm = (np.linalg.norm(POINTS[None, D['fg_idx'], :] - gctr[:, None, :], axis=2)
         / PT_S[None, D['fg_idx']])                     # (G,M) 归一化中心距离
print()
print(f'{"GT":>3s} {"尺寸":>7s} {"硬先验:框外数":>13s} {"硬先验:平均d/s":>15s} '
      f'{"软先验:框外数":>13s} {"软先验:平均d/s":>15s}')
for g in range(len(GT)):
    h, s_ = D['matching'][g], match_soft[g]
    print(f'{g:>3d} {GT_WH[g,0]:>4.0f}px {int((~D["both"][g,h]).sum()):>13d} '
          f'{dnorm[g,h].mean():>15.2f} {int((~D["both"][g,s_]).sum()):>13d} '
          f'{dnorm[g,s_].mean():>15.2f}')

assert dnorm[0, match_soft[0]].mean() < dnorm[0, D['matching'][0]].mean(),     '对 16px 的 GT，软先验应把正样本拉得离中心更近'
print()
print('✅ 练习 4 通过。诚实地读这张表：')
print('   · 对 16px 的 GT（先验内只有 2 个格点、k=6），软先验把正样本的平均距离从 0.89 拉到 0.74；')
print('   · 但 18px 的 GT 反而多了 2 个框外正样本 —— **软先验不是万能药**。')
print('   它做的事只有一件：把 0/100000 的**悬崖**换成**斜坡**。')
print('   悬崖的问题不是「让框外格点进来」，而是「进来的那些格点之间，距离信息被完全抹平」')
print('   （所有框外格点的惩罚项都是同一个 100000，排序只剩 cls+reg）。')
print('   斜坡则让排序始终携带距离信息 —— 这才是 RTMDet 换掉它的真正理由。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def matchable_size_range(anchor_side, thr=0.5):
    # min(L^2,s^2)/max(L^2,s^2) >= thr  <=>  sqrt(thr)*s <= L <= s/sqrt(thr)
    return anchor_side * np.sqrt(thr), anchor_side / np.sqrt(thr)

def is_matchable(gt_side, strides=STRIDES, anchor_scale=ANCHOR_SCALE, thr=0.5):
    for s in strides:
        lo, hi = matchable_size_range(anchor_scale * s, thr)
        if lo <= gt_side <= hi:
            return True
    return False

In [ ]:
# 练习 2 参考答案
def dynamic_k(ious, n_candidate_k=10, k_min=1):
    nk = min(n_candidate_k, ious.shape[1])
    topk = np.sort(ious, axis=1)[:, -nk:]
    return np.clip(np.floor(topk.sum(1)).astype(int), k_min, None)

In [ ]:
# 练习 3 参考答案
def assign_report(assign, gt_wh, pt_level, n_levels=3):
    G = len(gt_wh)
    pos = assign >= 0
    per_gt = [int((assign == g).sum()) for g in range(G)]
    per_level = [int((pos & (pt_level == l)).sum()) for l in range(n_levels)]
    return {
        'n_pos': int(pos.sum()),
        'per_gt': per_gt,
        'per_level': per_level,
        'starved': [g for g in range(G) if per_gt[g] <= 1],
        'imbalance': max(per_gt) / max(min(per_gt), 1),
    }

In [ ]:
# 练习 4 参考答案
def soft_center_prior(points, pt_stride, gt, radius=3.0):
    gctr = (gt[:, :2] + gt[:, 2:]) / 2                       # (G,2)
    d = np.linalg.norm(points[None, :, :] - gctr[:, None, :], axis=2)   # (G,N) 像素距离
    return 10.0 ** (d / pt_stride[None, :] - radius)

---
## 🧪 真实工程胶囊：mmdetection 里换分配器的完整配置与排查清单

In [ ]:
RECIPE = r'''
# ============ ① mmdetection：三种分配器的配置写法（可直接粘） ============
# ATSS（几何、稳定、无需 warmup，适合从零起训）
train_cfg = dict(assigner=dict(type='ATSSAssigner', topk=9))

# SimOTA（YOLOX；预测感知，密集监督）
train_cfg = dict(assigner=dict(
    type='SimOTAAssigner',
    center_radius=2.5,        # ← 密集小目标场景调小到 1.5，减少候选池重叠
    candidate_topk=10,        # ← dynamic-k 的 n_candidate_k
    iou_weight=3.0, cls_weight=1.0))

# RTMDet 的 DynamicSoftLabelAssigner（软标签 + 软中心先验，工业首选）
train_cfg = dict(assigner=dict(
    type='DynamicSoftLabelAssigner',
    topk=13,                  # dynamic-k 用 top-13 IoU 求和
    iou_weight=3.0,
    soft_center_radius=3.0))  # ← 10^(d/stride - 3)；小目标多可降到 2.5

# ============ ② 换分配器时必须一起改的三件事 ============
# 1. score 阈值要重扫：软标签让分类分数整体下移（目标值从 1.0 变成 IoU≈0.6~0.9）
#    直接沿用旧阈值 -> 召回莫名掉一截。用验证集扫 0.01~0.5 重定工作点。
# 2. 分类损失要配套：软标签必须用 QualityFocalLoss（能吃连续目标），
#    普通 FocalLoss 只接受 0/1 标签，配错会静默降点。
# 3. 预测感知的分配器需要 warmup：前 ~1-5 epoch 预测是噪声，代价矩阵没有意义。
#    RTMDet 的做法是初期整体 loss 权重小 + 用 AdamW 稳住；
#    某些实现直接前 N 个 epoch 用 ATSS，之后切 SimOTA。

# ============ ③ 分配相关的线上排查清单（按顺序做） ============
# [ ] 打印每个 GT 的正样本数（本 notebook 的 assign_report），找 starved 名单
# [ ] 按 GT 尺寸分桶统计正样本数：如果 <32px 桶的均值 < 2，先改架构不要改分配
# [ ] 打印 dynamic-k 的分布随 epoch 的变化：一直贴着 1 = 预测太差或 warmup 不够
# [ ] 统计去冲突仲裁次数：占正样本比例 > 10% 说明场景太密，考虑降 center_radius
# [ ] 检查是否有正样本落在 GT 框外（SimOTA 的已知边界行为）
# [ ] 换分配器后，**必须**重新扫 score 阈值再比 mAP，否则比的是阈值不是分配器

# ============ ④ 诊断代码片段（贴进 assigner 的 forward 末尾） ============
if self.debug and (self.iter % 200 == 0):
    n_pos_per_gt = matching_matrix.sum(1)                       # (G,)
    gt_area = (gt_bboxes[:, 2] - gt_bboxes[:, 0]) * (gt_bboxes[:, 3] - gt_bboxes[:, 1])
    small = gt_area < 32 ** 2
    print(f'[assign] pos={int(matching_matrix.sum())} '
          f'k_mean={dynamic_ks.float().mean():.2f} '
          f'starved={int((n_pos_per_gt <= 1).sum())} '
          f'small_gt_pos_mean={n_pos_per_gt[small].float().mean():.2f}')
'''
print(RECIPE)
for key in ['ATSSAssigner', 'SimOTAAssigner', 'DynamicSoftLabelAssigner',
            'center_radius', 'soft_center_radius', 'QualityFocalLoss',
            'starved', 'score 阈值要重扫']:
    assert key in RECIPE, key
print('✅ 配方覆盖：三种分配器配置 / 换分配器的连带改动 / 线上排查清单 / 诊断代码')

### 小结

- **标签分配决定训练信号的分布本身**。网络结构决定模型能表达什么，
  分配决定模型实际学到什么 —— 它不含一个参数，却值几个点 AP。
- **静态 IoU 阈值的根本问题是尺度不公平**：可匹配的 GT 尺寸区间是
  $[\sqrt{\tau}s,\ s/\sqrt{\tau}]$，落在区间外的目标**数学上不可能**被匹配到，
  只能靠兜底规则塞一个低质量 anchor —— 那是噪声监督。
- **ATSS 用 `mean + std` 把「选哪层」和「选几个」一次解决**。本课实测阈值从
  16px 的 0.21 到 48px 的 0.48，固定 0.5 相当于对小目标单方面提高 2.3 倍门槛。
- **SimOTA 四步：几何先验 → 代价矩阵 → dynamic-k → 去冲突**。dynamic-k 让配额
  随预测质量自动伸缩（早期≈1，后期 8–10），是它最漂亮的性质。
  代价：去冲突时**被抢走的名额不补**，密集场景下会削弱某个 GT。
- **TaskAligned 的 $t=s^{\alpha}u^{\beta}$ 中 $\beta \gg \alpha$ 是刻意的**：
  主要按定位质量选样本，分类分数只是修正项。同一个度量既选样本又当软标签。
- **RTMDet 用连续衰减的软中心先验替换 0/100000 悬崖**，避免 dynamic-k 顶穿硬先验
  去选框外格点。软标签的代价：分类分数不再是概率，**score 阈值必须重扫**。
- **TSR 的硬约束：16×16 的标志全图只有 2 个格点中心落在框内。**
  正样本上限是 stride 与输入分辨率决定的物理量，**分配算法突破不了**——
  必须先解决架构（P2 层 / 高分辨率 / ROI 裁剪），再谈分配。
- **一对一是推理端的需求，不是训练端的最优**。Group/H-/Co-DETR 训练时加回一对多分支
  就是证据；YOLOv10 的一致双分配是目前最干净的统一。

下一站：**模块 03 · RTMDet 解剖** —— 大核、共享头，以及本模块埋下的软标签代价函数的完整版。